# Redução de Dimensionalidade das Métricas Locais via PCA

In [35]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

### Carregando o dataset

In [36]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape
df_filtered = df_dataset.columns[df_dataset.columns.str.contains("age", case=False)]
df_filtered


Index(['age', 'age_25-30', 'age_30-35', 'age_55-60', 'age_60-65', 'age_65-70',
       'age_70-75', 'age_75-80'],
      dtype='object')

### Seleção Métricas Locais

In [38]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
}


len(NODE_FEATURES)

212

In [39]:
# descarta colunas sem informação (constantes ou duplicadas), se houver
DROP_REDUNDANT = True

_raw = df_dataset[NODE_FEATURES]
constant_cols = [c for c in NODE_FEATURES if _raw[c].nunique(dropna=True) <= 1]
_dup = _raw.T.duplicated()
duplicate_cols = [c for c, d in zip(NODE_FEATURES, _dup) if d and c not in constant_cols]

if DROP_REDUNDANT:
    NODE_FEATURES = [c for c in NODE_FEATURES if c not in constant_cols and c not in duplicate_cols]

print(f"Constantes: {len(constant_cols)} | duplicadas: {len(duplicate_cols)} | usadas no PCA: {len(NODE_FEATURES)}")
print("Tipos constantes:", sorted({c.rsplit("_", 1)[0] for c in constant_cols}))
print("Tipos duplicados:", sorted({c.rsplit("_", 1)[0] for c in duplicate_cols}))

Constantes: 0 | duplicadas: 0 | usadas no PCA: 212
Tipos constantes: []
Tipos duplicados: []


In [40]:
X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"].values
groups = df_dataset["subject_id"].values

print(f"{len(X)} registros de {df_dataset['subject_id'].nunique()} participantes (EO e EC de cada um)")

108 registros de 54 participantes (EO e EC de cada um)


### Padronização das Feaures

Imputação pela mediana: nem todo participante tem todos os eletrodos. Valor ausente significa eletrodo ausente, não conectividade zero, por isso não se usa 0.

In [41]:
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

In [42]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [43]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([-2.25128558e-16,  3.53626593e-16,  2.56996071e-17,  4.06053791e-17,
         6.78469626e-16]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [44]:
print("NaN antes da imputação:", int(df_dataset[NODE_FEATURES].isna().sum().sum()))
print("NaN depois da imputação:", int(np.isnan(X).sum()))

NaN antes da imputação: 224
NaN depois da imputação: 0


In [45]:
nan_per_feature = df_dataset[NODE_FEATURES].isna().sum()
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

hub_frequency_CP6    8
betweenness_CP6      8
clustering_CP6       8
degree_CP6           8
degree_CP1           6
                    ..
hub_frequency_C4     1
degree_F6            1
clustering_F6        1
betweenness_F6       1
hub_frequency_F6     1
Length: 80, dtype: int64

In [46]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

In [47]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})

In [48]:
df_var

,component,explained_variance,cumulative_variance
0,1,4.982362e-01,0.498236
1,2,3.959323e-02,0.537829
2,3,3.168277e-02,0.569512
3,4,2.440920e-02,0.593921
4,5,2.167651e-02,0.615598
...,...,...,...
103,104,8.080212e-05,0.999862
104,105,6.150028e-05,0.999924
105,106,4.641226e-05,0.999970
106,107,2.963564e-05,1.000000


In [49]:
n80 = int(np.argmax(cum_var >= 0.8) + 1)
n90 = int(np.argmax(cum_var >= 0.9) + 1)
print(f"PC1: {explained_var[0]:.1%} | PC2: {explained_var[1]:.1%}")
print(f"Componentes para 80%: {n80} | para 90%: {n90}")

PC1: 49.8% | PC2: 4.0%
Componentes para 80%: 19 | para 90%: 34


In [50]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()

Observa-se que 19 componentes principais são suficientes para explicar 80% da variância total dos dados, enquanto 34 componentes explicam 90%, indicando elevada redundância entre as métricas de rede (212 variáveis: grau ponderado normalizado, clustering ponderado, intermediação com distância 1/peso e frequência de hub, em 53 eletrodos).

### PCA com 2 componentes

In [51]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}


In [52]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y
df_pca["subject_id"] = groups

In [53]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()

Aqui você responde visualmente:

existe separação EO × EC? Não há separação clara entre EO e EC

existe sobreposição? Sim

clusters? Existem regiões de maior densidade, mas não clusters bem definidos.

A projeção das métricas locais no espaço definido pelos dois primeiros
componentes principais não evidenciou uma separação clara entre as
condições EO e EC. Observa-se uma ampla sobreposição entre os pontos
correspondentes às duas condições experimentais, indicando que os
padrões dominantes de variabilidade capturados pelos componentes
principais são compartilhados entre os grupos.

Embora seja possível identificar regiões de maior densidade de pontos,
não foram observados clusters bem definidos ou fronteiras visuais que
permitam distinguir claramente as condições experimentais no espaço
PC1–PC2.


### Contribuição das features

In [54]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)

In [55]:
loadings

,PC1,PC2
degree_AF3,0.084613,-0.036846
clustering_AF3,0.090912,-0.029498
betweenness_AF3,-0.036047,0.122457
hub_frequency_AF3,-0.018171,-0.084451
degree_AF4,0.083653,-0.077392
...,...,...
hub_frequency_Pz,-0.060472,-0.032516
degree_TP8,0.087853,0.031052
clustering_TP8,0.092974,0.015033
betweenness_TP8,-0.020639,0.018314


In [56]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
clustering_P3,0.096158,0.020989
clustering_P4,0.095840,0.029255
clustering_P5,0.095835,0.022536
clustering_P6,0.095811,0.029296
clustering_PO4,0.095722,0.022571
clustering_Oz,0.095419,0.028984
clustering_PO3,0.095363,0.024414
clustering_POz,0.095032,0.021309
clustering_Pz,0.095029,0.030945
clustering_PO8,0.095014,0.026663


In [57]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
betweenness_CP1,0.044442,0.206605
betweenness_C1,0.044495,0.206269
betweenness_CP4,0.048093,0.203760
betweenness_FC2,0.043063,0.173782
hub_frequency_FC3,0.021431,0.169275
betweenness_CP3,0.050143,0.164461
betweenness_C3,0.055023,0.161385
hub_frequency_Fz,0.017213,0.156974
betweenness_P1,0.051423,0.153563
betweenness_P2,0.044972,0.150598


#### Cargas por tipo de métrica

In [58]:
_lt = loadings[["PC1", "PC2"]].assign(tipo=[c.rsplit("_", 1)[0] for c in loadings.index])

# cada componente tem norma 1: a soma das cargas ao quadrado de um tipo é a fração do componente formada por ele
loadings_by_type = _lt.groupby("tipo").agg(
    n_variaveis=("PC1", "size"),
    PC1_contribuicao=("PC1", lambda s: (s ** 2).sum()),
    PC2_contribuicao=("PC2", lambda s: (s ** 2).sum()),
    PC1_carga_media=("PC1", "mean"),
    PC1_frac_positivas=("PC1", lambda s: (s > 0).mean()),
).sort_values("PC1_contribuicao", ascending=False)

fig = px.bar(
    loadings_by_type.reset_index().melt(
        id_vars="tipo", value_vars=["PC1_contribuicao", "PC2_contribuicao"],
        var_name="componente", value_name="contribuicao",
    ),
    x="tipo", y="contribuicao", color="componente", barmode="group",
    title="Contribuição de cada tipo de métrica para PC1 e PC2",
    labels={"tipo": "Métrica", "contribuicao": "Soma das cargas²", "componente": "Componente"},
)
fig.show()

loadings_by_type.round(3)

,n_variaveis,PC1_contribuicao,PC2_contribuicao,PC1_carga_media,PC1_frac_positivas
tipo,,,,,
clustering,53,0.455,0.045,0.093,1.000
degree,53,0.408,0.115,0.088,1.000
betweenness,53,0.076,0.414,-0.036,0.000
hub_frequency,53,0.061,0.425,-0.030,0.019


Quais métricas e nós mais contribuem para cada componente?

PC1 explica 49,8% da variância e PC2, 4,0%. PC1 é formado principalmente por clustering (≈46% da soma das cargas²) e grau (≈41%), com cargas positivas em todos os eletrodos; intermediação (≈8%) e frequência de hub (≈6%) entram com cargas negativas em quase todos. É um eixo de nível geral de sincronização: participantes cujos eletrodos passam mais tempo sincronizados têm também triângulos mais intensos, caminhos mais distribuídos (menos intermediação concentrada) e menos eletrodos se destacando como hubs. PC2 é dominado por frequência de hub (≈43%) e intermediação (≈41%), com sinais opostos entre regiões, e capta como os hubs e as pontes se distribuem na rede.

### PCA com mais componentes - Para ML

In [59]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(108, 34)

Para as análises de aprendizado de máquina, foi utilizado PCA com retenção de 90% da variância, resultando em 34 componentes principais.

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional indica padrões de organização distintos entre as
condições experimentais, embora com sobreposição parcial, sugerindo que métodos
não lineares ou classificadores supervisionados podem capturar diferenças mais sutis.

Os loadings indicam que métricas associadas a determinados nós apresentam maior
contribuição para os componentes principais, reforçando a relevância da análise
local da conectividade funcional.


-----

### Teste Estatístico

EO e EC são do mesmo participante, então a comparação de PC1 é **pareada** por `subject_id` (teste t pareado e Wilcoxon), e não entre grupos independentes.

In [60]:
pc1_wide = df_pca.pivot(index="subject_id", columns="condition", values="PC1").dropna()
pc1_eo, pc1_ec = pc1_wide["EO"], pc1_wide["EC"]

print(f"Pares completos: {len(pc1_wide)}")
print(f"Mediana da diferença EO - EC: {(pc1_eo - pc1_ec).median():.3f}")
ttest_rel(pc1_eo, pc1_ec)

Pares completos: 54
Mediana da diferença EO - EC: -2.309


TtestResult(statistic=np.float64(-2.327878928922566), pvalue=np.float64(0.023768020351339902), df=np.int64(53))

In [61]:
fig = px.line(
    df_pca.sort_values("condition", ascending=False),
    x="condition", y="PC1", line_group="subject_id", markers=True,
    title="PC1 por participante: EO vs EC",
    category_orders={"condition": ["EO", "EC"]},
)
fig.update_traces(opacity=0.4)
fig.show()

In [62]:
wilcoxon(pc1_eo, pc1_ec)

WilcoxonResult(statistic=np.float64(489.0), pvalue=np.float64(0.029058627528118094))

In [63]:
px.box(df_pca, x="condition", y="PC1")

In [64]:
px.box(df_pca, x="condition", y="PC2")

In [65]:
df_pca = df_pca[df_pca["condition"].isin(["EO", "EC"])]

In [66]:
cv_groups = GroupKFold(n_splits=5)

# PCA ajustado dentro de cada dobra; EO e EC do mesmo participante ficam sempre na mesma dobra
model_pca2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("clf", LogisticRegression()),
])

scores = cross_val_score(
    model_pca2, df_dataset[NODE_FEATURES], df_dataset["condition"],
    cv=cv_groups, groups=groups,
)
print(scores.round(3))
scores.mean()

[0.5   0.636 0.636 0.545 0.35 ]


np.float64(0.5336363636363636)

Com validação cruzada agrupada por participante (GroupKFold, 5 dobras) e o PCA ajustado dentro de cada dobra, a regressão logística com PC1 e PC2 obteve acurácia média de 53,4% (dobras entre 35% e 64%), pouco acima do acaso (50%). A capacidade discriminativa das duas primeiras componentes entre EO e EC é, portanto, fraca.

In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"]

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,  # equivalente a L1 (penalty='l1' deprecated)
                max_iter=5000,
            ),
        ),
    ]
)

In [68]:
scores = cross_val_score(model, X, y, cv=cv_groups, groups=groups)
print(scores.round(3))
scores.mean()

[0.545 0.591 0.364 0.591 0.5  ]


np.float64(0.5181818181818182)

Os dois primeiros componentes principais resumem boa parte da variabilidade das métricas de rede, mas não separam visualmente EO e EC no plano PC1–PC2, porque a variação entre participantes é muito maior que a diferença entre condições. Como EO e EC vêm dos mesmos 54 participantes, a comparação de PC1 foi feita de forma pareada: PC1 foi menor em EO do que em EC em 34 dos 54 participantes (mediana da diferença EO − EC = −2,31; teste t pareado: p = 0,024; Wilcoxon: p = 0,029). Ou seja, dentro de cada pessoa, o nível geral de sincronização tende a ser um pouco menor com olhos abertos. Essa diferença, porém, é pequena diante da variação entre pessoas: com validação agrupada por participante, a regressão logística com PC1 e PC2 teve acurácia de 53,4%, e o modelo L1 com todas as métricas, 51,8%, ambos próximos do acaso. A diferença EO − EC também depende da idade (ver `04_pca_metadata`).